# Постановка задачи.

Здесь мы генерируем последовательности в виде: последовательность-признаки (20) и ожидаемое предсказание на следующей позиции (21) - метка. Почследовательности иммитируют данные 4-х классов, и для каждого класса метка должна лежать в соответствующем, для данного класса, диапазоне чисел.  
Здесь все последовательности равной длины, хотя код предусматривает обработку последовательностей разной длины (не не более 20).

# Как решаем  (на PyTorch)

- Создадим класс TransformerEncoder с использованием позиционных кодировок.  
- Напишем функцию для обучения модели.  

- Для решения проблемы обработки последовательностей переменной длины используем паддинг с помощью pad_sequence. Используем collate_fn в DataLoader для обработки переменной длины. Передадим маску паддинга в TransformerEncoder для игнорирования паддинговых элементов.  
!!! Маскирование паддинга позволяет модели игнорировать паддинг-значения в процессе вычислениях внимания и Использовать разные длины последовательностей в одном батче без искажений.

# Логика работы кода

### Кратко об основных процедурах кода  
1. PositionalEncoding: Мы создаём класс для позиционных кодировок, которые добавляются к эмбеддингам. Эти кодировки содержат информацию о позиции каждого токена в последовательности, что помогает модели учитывать порядок чисел в последовательности.  

2. TransformerEncoder: Этот класс включает в себя слой эмбеддингов, слой позиционных кодировок, слой TransformerEncoder и полносвязный слой для предсказания.  

3. DataLoader: Создаём обучающий набор данных с 5 последовательностями, где каждой последовательности соответствует целевое значение (следующее число в последовательности).  

4. Обучение: Для обучения мы используем MSELoss как функцию потерь, так как задача — регрессия. Оптимизатор — Adam.  

5. Предсказание: После обучения модели, мы можем подать на неё новую последовательность и получить предсказание следующего числа.  

### Как работает модель  
> Модель обучается на последовательностях чисел, чтобы предсказать следующее значение на основе предыдущих чисел.  

> На каждом шаге обучения модель обновляет свои параметры (включая веса эмбеддингов и параметры self-attention), чтобы улучшить предсказания.  

### Что важно  
> __Позиционные кодировки__ помогают модели понимать порядок чисел в последовательности.  
> __Self-attention__ на основе Transformer позволяет учитывать связи между всеми токенами в последовательности, что важно для понимания контекста.

# Код

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.nn.utils.rnn import pad_sequence
import math

In [2]:
import random

## <font color='lightgreen'>Гиперпараметры

In [3]:
vocab_size = 100 # По умолчанию это означает, что словарь содержит индекся от 0 до 100
embedding_dim = 16
max_len = 20  # Максимальная длина последовательности
num_heads = 2
num_layers = 2
output_size = 1  # Целевое значение — это одно число

## <font color='lightgreen'>Данные для обучения

Генерируем последовательности (seq_len = 20), которые моделируют 4 класса.  
В 1 классе даннные sequence лежат в диапазоне от 0 до 30, target - от 30 до 40,  
во 2: sequence - от 20 до 50, target - от 50 до 60,  
в 3: sequence- от 40 до 70, target - от 70 до 80,  
в 4: sequence - от 60 до 90, target - от 90 до 99.  
Словарь vocab_size содержит 100 категорий.
Т.о., обучив модель на этих данных

In [5]:
# Генерация синтетических данных
def generate_data(num_samples=1000, seq_len=20):
    data = []
    for idx in range(num_samples):
        sequence = []

        # 1 часть: 0–30
        if  idx < num_samples // 4:
          sequence += [random.randint(0, 30) for _ in range(seq_len)]
          target = random.randint(30, 40)

        # 2 часть: 20–50
        elif idx < num_samples // 2:
          sequence += [random.randint(20, 50) for _ in range(seq_len)]
          target = random.randint(50, 60)

        # 3 часть: 40–70
        elif idx < (3 * num_samples // 4):
          sequence += [random.randint(40, 70) for _ in range(seq_len)]
          target = random.randint(70, 80)

        # 4 часть: 60–90
        else:
          sequence += [random.randint(60, 90) for _ in range(seq_len)]
          target = random.randint(90, 99)

        data.append((sequence, target))
    return data

In [6]:
# Кастомный Dataset
class CustomDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sequence, target = self.data[idx]
        return torch.tensor(sequence, dtype=torch.long), target  # <<< target оставляем просто int

In [7]:
# Генерация данных
data = generate_data(num_samples=1000, seq_len=20)

In [22]:
print(type(data))
print(data[:1])
print(data[250:251])
print(data[500:501])
print(data[750:751])
print(type(data[750:751]))

<class 'list'>
[([22, 28, 0, 13, 13, 24, 30, 6, 25, 11, 9, 8, 6, 12, 9, 14, 21, 18, 26, 9], 34)]
[([21, 40, 46, 35, 45, 24, 36, 26, 49, 39, 43, 39, 44, 47, 36, 29, 27, 45, 45, 43], 55)]
[([45, 54, 42, 56, 68, 51, 59, 58, 46, 67, 48, 41, 56, 70, 51, 58, 61, 58, 67, 69], 71)]
[([70, 65, 63, 67, 68, 79, 90, 68, 75, 72, 65, 61, 63, 79, 61, 73, 65, 63, 75, 84], 93)]
<class 'list'>


In [23]:
# Создание dataset - конкретного экземпляра класса CustomDataset
dataset = CustomDataset(data)

## <font color='lightgreen'>Функция для обработки последовательностей переменной длины

In [10]:
def collate_fn(batch):

  # Смотрим, какие типы данных в batch

    # print(f"Type of batch in collate_fn: {type(batch)}")
    # Правильно: <class 'list'> — это говорит о том,
    # что batch в collate_fn — это список, который содержит
    # последовательность и целевую метку.
    # print(f"First item in batch: {type(batch[0])}")
    # <class 'tuple'> — это кортеж, где первый элемент —
    # это сама последовательность, а второй — целевая метка.
    # print("_________________")

    # Распаковываем данные из batch
    sequences, targets = zip(*batch)

    # Преобразуем каждую последовательность в тензор (если это не тензоры)
    # sequences = [torch.tensor(seq) for seq in sequences]  # Преобразуем каждый элемент в тензор
    sequences = pad_sequence(sequences, batch_first=True, padding_value=0)
    targets = torch.tensor(targets, dtype=torch.long)  # Преобразуем цели в тензор


    # Проверим целевые метки на выход за пределы vocab_size
    assert targets.max() < vocab_size, f"Target {targets.max()} is out of bounds for vocab_size {vocab_size}"

    # Маска для паддинга (True — это паддинг, False — данные)
    padding_mask = sequences == 0  # Маска, где 0 — это паддинг

    # Проверим типы данных после преобразования
    # print(f"Type of sequences after padding: {type(sequences)}")
    # <class 'torch.Tensor'>
    # print(f"Type of targets: {type(targets)}")
    # <class 'torch.Tensor'>
    # print(f"Type of padding_mask: {type(padding_mask)}")
    # <class 'torch.Tensor'>
    # print("_________________")

    return sequences, targets, padding_mask

## <font color='lightgreen'>Создание загрузчика данных DataLoader

In [11]:
train_loader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

# Смотрим, правильность типов данных после загрузки
for batch in train_loader:
    print(f"Type of batch: {type(batch)}") # <class 'tuple'> — это уже кортеж, состоящий из трех элементов (последовательность, целевая метка и маска паддинга).
    print(f"Type of first element in batch: {type(batch[0])}") # class 'torch.Tensor'> — это тензор последовательности.
    print("_________________")

    sequences, targets, padding_mask = batch  # Извлекаем данные

    print(f"Sequences shape: {sequences.shape}") # Ожидаем [batch_size, seq_len], torch.Size([5, 5]) — это предполагаемый размер батча: 5 последовательностей, каждая из которых имеет длину 5.
    print(f"Targets shape: {targets.shape}") # Ожидаем [batch_size], torch.Size([5]) — целевые метки для каждого из элементов батча.
    print(f"Padding mask shape: {padding_mask.shape}") # Ожидаем [batch_size, seq_len], torch.Size([5, 5]) — маска паддинга имеет такую же форму, как и последовательности, где каждый элемент может быть True (если это паддинг) или False (если это настоящий элемент).

    break  # Выводим только первую партию данных

Type of batch: <class 'tuple'>
Type of first element in batch: <class 'torch.Tensor'>
_________________
Sequences shape: torch.Size([32, 20])
Targets shape: torch.Size([32])
Padding mask shape: torch.Size([32, 20])


## <font color='lightgreen'>PositionalEncoding:</font>  
Мы создаём класс для позиционных кодировок, которые добавляются к эмбеддингам. Эти кодировки содержат информацию о позиции каждого токена в последовательности, что помогает модели учитывать порядок чисел в последовательности.

In [12]:
# Класс позиционной кодировки
class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()

        self.encoding = torch.zeros(max_len, d_model)

        # Генерация позиционных кодировок
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(torch.log(torch.tensor(10000.0)) / d_model))

        self.encoding[:, 0::2] = torch.sin(position * div_term)
        self.encoding[:, 1::2] = torch.cos(position * div_term)
        self.encoding = self.encoding.unsqueeze(0)

  def forward(self, x):
        return x + self.encoding[:, :x.size(1)]

## про __class PositionalEncoding(nn.Module)__  

<font color='lightgreen'>__nn.Module__</font> — это базовый класс любой нейросетевой архитектуры в PyTorch.
Создание своих моделей происходит методом наследования от него.  
 <font color='lightgreen'>**__ init __**</font> - создаём компоненты модели  

1. __Позиции (индексы токенов в последовательности)__  
> <font color='lightgreen'>position = torch.arange(0, max_len).unsqueeze(1)</font>  # shape: [max_len, 1]  
Создаёт тензор с числами от 0 до max_len - 1, по одной позиции в строке.  
>> <font color='lightgreen'>.unsqueeze(1)</font> делает из [max_len] -> [max_len, 1], чтобы можно было делать поблочное умножение.  
<font color='lightgreen'> </font>

2. __Делители (div_term)__  
<font color='lightgreen'>div_term = torch.exp(torch.arange(0, embedding_dim, 2) * -(torch.log(torch.tensor(10000.0)) / embedding_dim))</font>  
Это "частота" для синуса и косинуса. Чем дальше по координате эмбеддинга, тем меньше шаг (частота выше). Каждая чётная координата в эмбеддинге будет иметь свою уникальную частоту.  

3. __Инициализация pe__ — positional encoding матрицы
<font color='lightgreen'>pe = torch.zeros(max_len, embedding_dim) </font>
Матрица, в которую запишутся значения синуса и косинуса.  

4. __Запись синуса и косинуса__  
<font color='lightgreen'>pe[:, 0::2] = torch.sin(position * div_term)  
pe[:, 1::2] = torch.cos(position * div_term)</font>  
Для каждой строки (позиции) мы заполняем:
> - чётные координаты — sin(position * frequency)  
> - нечётные координаты — cos(position * frequency)  
Таким образом, каждое положение в последовательности получает уникальный вектор, но с одинаковой длиной, как и эмбеддинг.  

5. __Добавление размерности батча__  
<font color='lightgreen'>pe = pe.unsqueeze(0)  # shape: [1, max_len, embedding_dim]</font>  
Нам нужно иметь размерность [batch_size, seq_len, embedding_dim].  
Мы делаем unsqueeze(0), чтобы потом можно было сложить с входом x.  

6. __register_buffer__  
<font color='lightgreen'>self.register_buffer('pe', pe)</font>  
pe сохраняется в модели как буфер, а не обучаемый параметр. Он будет сохранён вместе с моделью (state_dict), но не обновляется градиентами.  

7. __Forward: добавление к эмбеддингам__  
<font color='lightgreen'>return x + self.pe[:, :x.size(1)]</font>  
x — это результат embedding(x), размер [batch_size, seq_len, embedding_dim].  
self.pe[:, :x.size(1)] выбирает нужное число позиций (вдруг последовательность короче max_len).  
Далее просто складываем поэлементно эмбеддинг и позиционную кодировку.  


__Итого__  
Мы имеем представление, например, входного вектора в виде матрицы:
где каждая строка — это один токен (или число) в последовательности,
а каждый столбец — это одна координата эмбеддинга этого токена.  

x = [[0.1, 0.2, 0.3],     позиция 0  
     [0.4, 0.5, 0.6]]     позиция 1  

А pe:  
pe = [[0.01, 0.02, 0.03],  позиция 0  
      [0.04, 0.05, 0.06]]  позиция 1  

То результат будет:
x + pe = [[0.11, 0.22, 0.33],
          [0.44, 0.55, 0.66]]
          
<font color='lightgreen'></font>


<font color='lightgreen'> </font>

##  <font color='lightgreen'>TransformerEncoder</font>

2. 🔧 Цель класса <font color='lightgreen'>TransformerEncoder</font>  
Класс реализует односторонний (encoder-only) блок трансформера, который:  
- получает на вход последовательность чисел (категорий),  
- превращает каждую категорию в эмбеддинг,  
- добавляет позиционную информацию,  
- пропускает это всё через один или несколько слоёв self-attention,  
- возвращает выход, с которым можно работать: делать классификацию, предсказания и т.д.

In [13]:
# 2. Класс Transformer Encoder для предсказания следующего числа-категории
class TransformerEncoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, max_len, num_heads, num_layers, output_size):
        super(TransformerEncoder, self).__init__()

        # Параметры модели
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.pos_encoder = PositionalEncoding(embedding_dim, max_len)

        # Transformer Encoder
        self.transformer_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embedding_dim, nhead=num_heads),
            num_layers=num_layers
        )

        # Полносвязный слой для предсказания
        self.fc = nn.Linear(embedding_dim, vocab_size)

  def forward(self, x, padding_mask):
        # Применение эмбеддингов
        x = self.embedding(x)   # [batch_size, seq_len, embedding_dim]
        # Применение позиционной кодировки
        x = self.pos_encoder(x)

        # x = x.transpose(0, 1)  # Теперь (seq_len, batch_size, embedding_dim)

        # Пропускаем через TransformerEncoder, используя маску паддинга
        output = self.transformer_encoder(x, src_key_padding_mask=padding_mask)

        # output = output.transpose(0, 1)  # (batch_size, seq_len, embedding_dim)

        last_token = output[:, -1, :]       # берём выход с последней позиции
        logits = self.fc(last_token)        # [batch_size, vocab_size]

        return logits

## про __class TransformerEncoder(nn.Module)__  

 <font color='lightgreen'>__nn.Module__</font> — это базовый класс любой нейросетевой архитектуры в PyTorch.
Создание своих моделей происходит методом наследования от него.  
<font color='lightgreen'>**__ init __**</font>: создаём компоненты модели.  

🔹 <font color='lightgreen'>self.embedding = nn.Embedding(vocab_size, embedding_dim)</font>  
> - Преобразует числа (индексы категорий) во вектора фиксированной длины embedding_dim. Например: [7, 25, 3] → [[0.2, 0.5], [0.1, 0.4], [0.9, 0.7]]  

🔹<font color='lightgreen'>self.pos_encoder = PositionalEncoding(...)</font>  
> - Добавляет информацию о порядке токенов, потому что nn.Embedding сам по себе просто смотрит на значения, но не на их позиции.  

🔹<font color='lightgreen'>encoder_layer = nn.TransformerEncoderLayer(...)  
🔹self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=...)</font>  
> - Один слой self-attention: attention + feedforward + нормализация.  
> - Обернутый в nn.TransformerEncoder он повторяется num_layers раз.

🔹<font color='lightgreen'>def forward(self, x)</font>  
> - <font color='lightgreen'>x = self.embedding(src)</font> Преобразуем индексы в эмбеддинги.  
> - <font color='lightgreen'>x = self.pos_encoder(x)</font> Добавляем позиционную кодировку.  
> - <font color='lightgreen'>output = self.transformer_encoder(x)</font> Пропускаем через трансформер.
> - <font color='lightgreen'>logits = self.fc_out(output[:, -1, :])</font>  Предсказание следующего токена  Мы берём выход последнего токена в последовательности, потому что мы хотим предсказать следующее значение.

🔚 Вывод  
Этот класс подходит для задачи:  
> - "предсказать следующее значение в числовой последовательности".  
> - Учитывает порядок (позиционка).  
> - Гибкий: можно управлять размерностью, числом голов внимания, количеством слоёв.

## <font color='lightgreen'>Инициализация модели, настройка оптимизатора и функции потерь

In [14]:
# Инициализация модели
model = TransformerEncoder(vocab_size, embedding_dim, max_len, num_heads, num_layers, output_size)

/opt/anaconda3/lib/python3.11/site-packages/torch/nn/modules/transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


In [15]:
# Настройка оптимизатора и функции потерь
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

## про loss

Мы выбрали критерий по рассчету ошибок (loss),
loss — это суммарная ошибка классификации по батчу (или по всей выборке, если без батчей).  

📉 2. Каким должен быть нормальный loss?  
Для nn.CrossEntropyLoss():  
- Хороший loss обычно < 1.0 на один пример  
- На старте (если случайные веса) бывает 3–6  
Если loss растёт и не падает — модель не учится  

📌 Возможные причины большого loss:  
Модель не обучается:  
- lr слишком большая/маленькая  
- веса не обновляются  
- плохая инициализация  
- Целевая переменная не подготовлена правильно  
>> например, target имеет форму [batch_size, 1], а должен быть [batch_size] (1D целые индексы)  
- vocab_size не совпадает с количеством классов в fc  
- Проблемы с входами/эмбеддингами, например, входные значения за пределами [0, vocab_size - 1]  

✨     
__Loss-функция__	- nn.CrossEntropyLoss()  
__logits shape__	[- batch_size, vocab_size]  
__target shape__	- [batch_size]  
__target тип__	- LongTensor  
Пример	_loss = criterion(outputs, targets.squeeze())__

In [16]:
print(targets.max())  # Проверим максимальное значение в targets

tensor(99)


## <font color='lightgreen'>Обучение модели

In [17]:
# sequences.shape
padding_mask.shape

torch.Size([32, 20])

In [21]:
sequences

tensor([[64, 68, 57, 61, 56, 58, 50, 56, 46, 41, 61, 53, 70, 63, 57, 45, 46, 69,
         68, 47],
        [21, 18,  6, 23, 17, 26, 20, 12, 25, 17,  6, 24, 20, 30, 15, 28,  2, 21,
          0,  1],
        [70, 62, 56, 45, 62, 56, 59, 61, 56, 46, 69, 63, 62, 64, 55, 42, 69, 60,
         41, 45],
        [77, 67, 84, 63, 72, 85, 82, 62, 69, 77, 78, 85, 61, 87, 76, 80, 74, 74,
         73, 60],
        [42, 51, 69, 54, 56, 66, 55, 68, 64, 52, 69, 47, 67, 69, 66, 49, 47, 58,
         44, 48],
        [80, 90, 65, 73, 81, 82, 80, 70, 60, 74, 88, 81, 79, 60, 78, 65, 71, 64,
         82, 87],
        [65, 49, 66, 53, 40, 60, 45, 55, 70, 62, 41, 59, 55, 53, 40, 61, 57, 56,
         62, 66],
        [28, 20, 28,  0, 24, 22,  5, 23, 21, 21, 12, 29, 26,  5, 23,  1, 11, 16,
         22,  1],
        [65, 87, 72, 69, 79, 82, 89, 90, 61, 79, 64, 87, 68, 80, 62, 72, 89, 81,
         68, 86],
        [65, 78, 62, 68, 61, 82, 63, 89, 75, 73, 77, 67, 69, 70, 86, 69, 64, 79,
         79, 89],
        [8

In [18]:
num_epochs = 200

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for batch in train_loader:
        sequences, targets, padding_mask = batch

        optimizer.zero_grad()

        # Прямой проход
        outputs = model(sequences, padding_mask)
        # print(outputs)

        # Преобразуем targets в одномерный тензор (если это необходимо)
        targets = targets.view(-1)  # Преобразуем targets в форму [batch_size * seq_len]
        # print(targets)
        # Проверка формы outputs и targets
        # assert outputs.size(2) == vocab_size, f"Output vocab size mismatch: {outputs.size(2)} != {vocab_size}"
        # assert targets.size(0) == outputs.size(0) * outputs.size(1), f"Target size mismatch: {targets.size(0)} != {outputs.size(0)} * {outputs.size(1)}"

        # Вычисление потерь
        loss = criterion(outputs.view(-1, outputs.size(-1)), targets)  # Используем CrossEntropy
        # outputs.view(-1, outputs.size(-1)): это изменение формы тензора
        # outputs, чтобы он соответствовал формату, необходимому
        # для CrossEntropyLoss (нужно, чтобы его форма была
        #  [batch_size * seq_len, vocab_size]).
        loss.backward()
        optimizer.step()
#
        epoch_loss += loss.item()
#
    print(f'Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_loss / len(train_loader):.4f}')


AssertionError: expecting key_padding_mask shape of (20, 32), but got torch.Size([32, 20])

## <font color='lightgreen'>Оценка модели

Смотрим попадание в класс 1 (т. е., целевая метка должна попасть в д-н (30-40)

In [92]:
model.eval() # Перевод модели в режим оценки с помощью model.eval()
with torch.no_grad(): # Отключение градиентов
    test_seq = torch.randint(0, 30, (1, 20))  # генерируем последовательность, близкую к классу 1 (0,30)

    # Паддинг до максимальной длины (если последовательность короче max_len)
    seq_len = test_seq.size(1)
    if seq_len < max_len:
        # Паддинг с значением 0 (предположим, что 0 — это паддинг)
        padded_test_seq = torch.cat([test_seq, torch.zeros(1, max_len - seq_len, dtype=torch.long)], dim=1)
        # Маска для паддинга
        padding_mask = padded_test_seq == 0
    else:
        padded_test_seq = test_seq
        padding_mask = torch.zeros_like(padded_test_seq, dtype=torch.bool)  # Нет паддинга, все False

    # Применение модели для предсказания
    logits = model(padded_test_seq, padding_mask)

    # Предсказание (выход модели)
    predicted_value = logits.squeeze()  # Сжимаем лишние размеры
    print(predicted_value)

tensor([-7.4779, -6.9468, -7.3434, -8.4333, -4.8937, -7.4540, -8.2412, -9.1001,
        -7.3649, -7.0889, -8.2760, -6.3953, -7.6819, -8.7502, -8.4268, -7.4480,
        -7.3733, -5.1428, -9.1736, -6.0593, -7.3487, -7.5519, -7.2197, -5.7768,
        -7.3771, -7.8857, -7.5743, -8.2607, -8.9522, -7.6089,  4.4136,  6.1808,
        10.9454,  2.9027,  4.7592,  6.6769,  0.5993,  7.8784,  9.3610,  5.1108,
         5.7418, -8.1150, -6.4899, -8.7762, -7.9560, -7.6739, -6.5328, -7.8096,
        -6.2560, -7.5643, -4.4106, -3.4614, -3.6438,  0.4573,  0.0409,  1.2620,
        -6.6507, -4.8464, -3.5425, -2.2262,  2.3195, -9.5591, -8.6255, -7.3828,
        -6.4328, -9.2185, -8.9731, -8.7618, -8.3808, -6.6806,  0.0251,  2.5449,
        -2.4871,  4.3187,  3.5037,  2.5518, -2.6071,  4.0192,  4.9362,  0.1837,
         2.2693, -7.2676, -8.9560, -8.1926, -7.2610, -8.1917, -9.3886, -6.4312,
        -9.1402, -8.1375, -8.4498, -4.3129, -9.4370, -5.2151, -1.4339, -2.6554,
        -2.2152, -5.4064, -6.4466, -6.00

In [93]:
# Полученные вероятности (отнесения предсказанного значения к одному из классов, у нас их - 50, т.е. 50 категорий, размеченных от 0 до 50)
print("_______________________________")
probs = torch.softmax(predicted_value, dim=-1)
print("Вероятности:\n", probs)
print("_______________________________")
# Посмотреть топ-кандидатов (например, 3-х):

topk_probs, topk_indices = torch.topk(probs, k=3)
print("Топ-3 вероятности:\n", topk_probs)
print("Топ-3 индексы:\n", topk_indices)
print("_______________________________")

# Получить предсказанную категорию

predicted_class = predicted_value.argmax(dim=-1)
print("Предсказанный класс:\n", predicted_class)

_______________________________
Вероятности:
 tensor([7.7154e-09, 1.3123e-08, 8.8258e-09, 2.9678e-09, 1.0225e-07, 7.9020e-09,
        3.5964e-09, 1.5236e-09, 8.6381e-09, 1.1384e-08, 3.4732e-09, 2.2779e-08,
        6.2913e-09, 2.1617e-09, 2.9870e-09, 7.9494e-09, 8.5664e-09, 7.9701e-08,
        1.4156e-09, 3.1875e-08, 8.7798e-09, 7.1652e-09, 9.9882e-09, 4.2281e-08,
        8.5333e-09, 5.1318e-09, 7.0065e-09, 3.5268e-09, 1.7664e-09, 6.7677e-09,
        1.1266e-03, 6.5957e-03, 7.7352e-01, 2.4865e-04, 1.5916e-03, 1.0832e-02,
        2.4846e-05, 3.6018e-02, 1.5864e-01, 2.2624e-03, 4.2521e-03, 4.0799e-09,
        2.0722e-08, 2.1063e-09, 4.7833e-09, 6.3420e-09, 1.9852e-08, 5.5372e-09,
        2.6184e-08, 7.0766e-09, 1.6575e-07, 4.2827e-07, 3.5683e-07, 2.1555e-05,
        1.4214e-05, 4.8199e-05, 1.7643e-08, 1.0720e-07, 3.9488e-07, 1.4728e-06,
        1.3877e-04, 9.6276e-10, 2.4489e-09, 8.4849e-09, 2.1939e-08, 1.3534e-09,
        1.7298e-09, 2.1368e-09, 3.1277e-09, 1.7124e-08, 1.3992e-05, 1.7386

Смотрим попадание в класс 2 (т. е., целевая метка должна попасть в д-н (50-60)

In [94]:
model.eval() # Перевод модели в режим оценки с помощью model.eval()
with torch.no_grad(): # Отключение градиентов
    test_seq = torch.randint(20, 50, (1, 20))  # генерируем последовательность, близкую к классу 2 (20,50)

    # Паддинг до максимальной длины (если последовательность короче max_len)
    seq_len = test_seq.size(1)
    if seq_len < max_len:
        # Паддинг с значением 0 (предположим, что 0 — это паддинг)
        padded_test_seq = torch.cat([test_seq, torch.zeros(1, max_len - seq_len, dtype=torch.long)], dim=1)
        # Маска для паддинга
        padding_mask = padded_test_seq == 0
    else:
        padded_test_seq = test_seq
        padding_mask = torch.zeros_like(padded_test_seq, dtype=torch.bool)  # Нет паддинга, все False

    # Применение модели для предсказания
    logits = model(padded_test_seq, padding_mask)

    # Предсказание (выход модели)
    predicted_value = logits.squeeze()  # Сжимаем лишние размеры
    print(predicted_value)

tensor([-3.0677, -2.2407, -3.0673, -3.7035, -1.1485, -3.2576, -2.9852, -3.3366,
        -3.6137, -2.4627, -2.5921, -2.8301, -3.3810, -3.5368, -3.5978, -3.1545,
        -3.1195, -2.3453, -3.8982, -2.3763, -3.7657, -2.8234, -2.2859, -2.2726,
        -3.1631, -2.9472, -2.7446, -2.9247, -4.0022, -3.1563, -3.5072, -2.1006,
        -2.3122, -2.7733,  2.1288,  1.2490,  0.8604, -2.2373, -0.9488,  2.9424,
        -1.9939, -3.1851, -2.7545, -3.4991, -2.7579, -3.0834, -2.2044, -2.7757,
        -2.0738, -3.7891,  8.4270, 12.0650, 10.8539,  9.4474,  4.3436,  7.0663,
         5.6797,  5.1468, 11.8020,  5.5063,  5.2675, -4.1608, -3.9653, -2.5096,
        -2.3505, -2.9354, -3.7786, -3.7689, -2.8266, -2.8938,  0.1942,  0.4609,
        -4.6948, -1.5973, -1.2122, -2.6843, -2.8840, -7.9942, -3.6018, -5.4697,
        -0.1872, -2.2696, -4.1265, -2.2841, -2.5046, -2.8804, -3.9037, -2.2873,
        -3.8058, -3.6735, -6.4030, -5.2028,  2.8075,  2.0417, -3.1182, -7.2960,
        -4.3358,  2.8100, -0.8725, -2.02

In [95]:
# Полученные вероятности (отнесения предсказанного значения к одному из классов
print("_______________________________")
probs = torch.softmax(predicted_value, dim=-1)
print("Вероятности:\n", probs)
print("_______________________________")
# Посмотреть топ-кандидатов (например, 3-х):

topk_probs, topk_indices = torch.topk(probs, k=3)
print("Топ-3 вероятности:\n", topk_probs)
print("Топ-3 индексы:\n", topk_indices)
print("_______________________________")

# Получить предсказанную категорию

predicted_class = predicted_value.argmax(dim=-1)
print("Предсказанный класс:\n", predicted_class)

_______________________________
Вероятности:
 tensor([1.2295e-07, 2.8113e-07, 1.2301e-07, 6.5102e-08, 8.3795e-07, 1.0169e-07,
        1.3352e-07, 9.3963e-08, 7.1223e-08, 2.2517e-07, 1.9783e-07, 1.5594e-07,
        8.9881e-08, 7.6917e-08, 7.2366e-08, 1.1273e-07, 1.1674e-07, 2.5321e-07,
        5.3586e-08, 2.4548e-07, 6.1180e-08, 1.5698e-07, 2.6871e-07, 2.7230e-07,
        1.1176e-07, 1.3870e-07, 1.6984e-07, 1.4185e-07, 4.8295e-08, 1.1253e-07,
        7.9229e-08, 3.2341e-07, 2.6173e-07, 1.6505e-07, 2.2210e-05, 9.2143e-06,
        6.2471e-06, 2.8208e-07, 1.0233e-06, 5.0105e-05, 3.5981e-07, 1.0934e-07,
        1.6817e-07, 7.9873e-08, 1.6761e-07, 1.2104e-07, 2.9153e-07, 1.6465e-07,
        3.3220e-07, 5.9764e-08, 1.2074e-02, 4.5898e-01, 1.3671e-01, 3.3496e-02,
        2.0344e-04, 3.0967e-03, 7.7391e-04, 4.5419e-04, 3.5283e-01, 6.5068e-04,
        5.1247e-04, 4.1213e-08, 5.0108e-08, 2.1484e-07, 2.5189e-07, 1.4034e-07,
        6.0398e-08, 6.0984e-08, 1.5647e-07, 1.4630e-07, 3.2089e-06, 4.1899

Смотрим попадание в класс 3 (т. е., целевая метка должна попасть в д-н (70-80)

In [96]:
model.eval() # Перевод модели в режим оценки с помощью model.eval()
with torch.no_grad(): # Отключение градиентов
    test_seq = torch.randint(40, 70, (1, 20))  # генерируем последовательность, близкую к классу 3 (40,70)

    # Паддинг до максимальной длины (если последовательность короче max_len)
    seq_len = test_seq.size(1)
    if seq_len < max_len:
        # Паддинг с значением 0 (предположим, что 0 — это паддинг)
        padded_test_seq = torch.cat([test_seq, torch.zeros(1, max_len - seq_len, dtype=torch.long)], dim=1)
        # Маска для паддинга
        padding_mask = padded_test_seq == 0
    else:
        padded_test_seq = test_seq
        padding_mask = torch.zeros_like(padded_test_seq, dtype=torch.bool)  # Нет паддинга, все False

    # Применение модели для предсказания
    logits = model(padded_test_seq, padding_mask)

    # Предсказание (выход модели)
    predicted_value = logits.squeeze()  # Сжимаем лишние размеры
    print(predicted_value)

tensor([-0.9174, -1.9415, -2.3726, -2.0204, -2.0795, -1.5197, -2.4030, -1.9179,
        -3.1867, -1.8875, -1.7092, -0.8475, -1.7368, -3.0330, -2.3957, -1.1114,
        -1.9119, -1.8682, -2.5884, -0.4444, -1.3701, -1.8544, -1.4639, -1.1319,
        -2.3116, -3.2832, -1.9306, -1.6605, -2.2089, -1.5258, -2.1637, -6.0846,
        -1.3144,  2.1559, -3.2529, -3.2125, -2.1555,  0.2840,  0.6665,  1.0235,
         3.6152, -1.6629, -2.3642, -2.1392, -1.6864, -1.9713, -1.3233, -1.5512,
        -1.4580, -2.4534, -1.3615,  1.4803, -6.6371, -5.8460, -5.5357, -2.6895,
        -1.8402,  0.0638, -4.6693,  0.5287, -0.2068, -2.0992, -2.9919, -1.8419,
        -3.4969, -2.1560, -2.2121, -2.9400, -1.5326, -1.4748,  4.2355,  3.0813,
        12.5350,  5.0692, 11.8030,  0.0436,  8.1951,  3.1150, -0.1853,  6.6469,
         2.1165, -1.8363, -1.8658, -1.8707, -1.9859, -2.1011, -2.5644, -2.0271,
        -2.4873, -2.5146, -0.9189, -6.0601, -5.9005, -1.5540,  1.2131, -1.7946,
        -4.4012, -7.9806, -2.2774, -0.97

In [97]:
# Полученные вероятности (отнесения предсказанного значения к одному из классов
print("_______________________________")
probs = torch.softmax(predicted_value, dim=-1)
print("Вероятности:\n", probs)
print("_______________________________")
# Посмотреть топ-кандидатов (например, 3-х):

topk_probs, topk_indices = torch.topk(probs, k=3)
print("Топ-3 вероятности:\n", topk_probs)
print("Топ-3 индексы:\n", topk_indices)
print("_______________________________")

# Получить предсказанную категорию

predicted_class = predicted_value.argmax(dim=-1)
print("Предсказанный класс:\n", predicted_class)

_______________________________
Вероятности:
 tensor([9.5983e-07, 3.4467e-07, 2.2398e-07, 3.1854e-07, 3.0025e-07, 5.2553e-07,
        2.1727e-07, 3.5292e-07, 9.9225e-08, 3.6380e-07, 4.3481e-07, 1.0293e-06,
        4.2299e-07, 1.1571e-07, 2.1886e-07, 7.9053e-07, 3.5503e-07, 3.7089e-07,
        1.8050e-07, 1.5402e-06, 6.1033e-07, 3.7606e-07, 5.5569e-07, 7.7452e-07,
        2.3807e-07, 9.0097e-08, 3.4847e-07, 4.5651e-07, 2.6380e-07, 5.2232e-07,
        2.7600e-07, 5.4716e-09, 6.4528e-07, 2.0744e-05, 9.2871e-08, 9.6702e-08,
        2.7829e-07, 3.1910e-06, 4.6781e-06, 6.6852e-06, 8.9263e-05, 4.5543e-07,
        2.2587e-07, 2.8284e-07, 4.4485e-07, 3.3457e-07, 6.3960e-07, 5.0927e-07,
        5.5898e-07, 2.0658e-07, 6.1562e-07, 1.0555e-05, 3.1489e-09, 6.9458e-09,
        9.4725e-09, 1.6314e-07, 3.8144e-07, 2.5605e-06, 2.2528e-08, 4.0760e-06,
        1.9535e-06, 2.9440e-07, 1.2057e-07, 3.8079e-07, 7.2761e-08, 2.7814e-07,
        2.6297e-07, 1.2699e-07, 5.1880e-07, 5.4967e-07, 1.6599e-04, 5.2333

Смотрим попадание в класс 4 (т. е., целевая метка должна попасть в д-н (90-99)

In [98]:
model.eval() # Перевод модели в режим оценки с помощью model.eval()
with torch.no_grad(): # Отключение градиентов
    test_seq = torch.randint(60, 90, (1, 20))  # генерируем последовательность, близкую к классу 4 (60,90)

    # Паддинг до максимальной длины (если последовательность короче max_len)
    seq_len = test_seq.size(1)
    if seq_len < max_len:
        # Паддинг с значением 0 (предположим, что 0 — это паддинг)
        padded_test_seq = torch.cat([test_seq, torch.zeros(1, max_len - seq_len, dtype=torch.long)], dim=1)
        # Маска для паддинга
        padding_mask = padded_test_seq == 0
    else:
        padded_test_seq = test_seq
        padding_mask = torch.zeros_like(padded_test_seq, dtype=torch.bool)  # Нет паддинга, все False

    # Применение модели для предсказания
    logits = model(padded_test_seq, padding_mask)

    # Предсказание (выход модели)
    predicted_value = logits.squeeze()  # Сжимаем лишние размеры
    print(predicted_value)

tensor([ -3.5952,  -3.6584,  -3.6418,  -4.5298,  -1.7442,  -4.2718,  -4.2171,
         -4.3543,  -3.6684,  -3.6112,  -4.7177,  -3.0980,  -4.2002,  -4.2395,
         -3.8927,  -3.3943,  -3.5682,  -2.8534,  -4.9905,  -3.2861,  -2.6065,
         -3.3172,  -3.3090,  -2.5863,  -3.1850,  -4.3863,  -4.1466,  -3.9772,
         -4.9909,  -3.3829,  -2.4659,   2.0763,  -3.6016,   2.0915,  -0.6905,
          0.4994,   0.1354,  -4.4248,   1.4307,   5.4609,  -5.2464,  -4.7303,
         -2.7078,  -4.1923,  -3.4402,  -4.0532,  -2.8525,  -4.4764,  -2.6593,
         -3.6001,  -3.6252,  -6.6341,  -6.2254,  -7.6924,  -5.8228,  -5.2125,
         -5.8110,  -9.2020,  -2.5005, -13.1554,  -2.9523,  -4.8574,  -5.1655,
         -2.8046,  -3.6091,  -4.5027,  -4.9110,  -4.3284,  -3.4561,  -2.2775,
         -3.8368,   6.5407,   1.2882,   3.3831,  -0.7437,  -3.2301,   0.5118,
          1.4574,  -0.3945,  -4.3758,   2.0222,  -2.5804,  -3.9269,  -3.5207,
         -3.7201,  -3.0890,  -4.8222,  -3.3645,  -5.0555,  -3.61

In [99]:
# Полученные вероятности (отнесения предсказанного значения к одному из классов
print("_______________________________")
probs = torch.softmax(predicted_value, dim=-1)
print("Вероятности:\n", probs)
print("_______________________________")
# Посмотреть топ-кандидатов (например, 3-х):

topk_probs, topk_indices = torch.topk(probs, k=3)
print("Топ-3 вероятности:\n", topk_probs)
print("Топ-3 индексы:\n", topk_indices)
print("_______________________________")

# Получить предсказанную категорию

predicted_class = predicted_value.argmax(dim=-1)
print("Предсказанный класс:\n", predicted_class)

_______________________________
Вероятности:
 tensor([5.5334e-08, 5.1941e-08, 5.2812e-08, 2.1731e-08, 3.5226e-07, 2.8129e-08,
        2.9709e-08, 2.5900e-08, 5.1427e-08, 5.4454e-08, 1.8008e-08, 9.0972e-08,
        3.0214e-08, 2.9052e-08, 4.1095e-08, 6.7642e-08, 5.6845e-08, 1.1618e-07,
        1.3709e-08, 7.5372e-08, 1.4873e-07, 7.3066e-08, 7.3669e-08, 1.5175e-07,
        8.3396e-08, 2.5084e-08, 3.1879e-08, 3.7765e-08, 1.3704e-08, 6.8418e-08,
        1.7117e-07, 1.6072e-05, 5.4977e-08, 1.6318e-05, 1.0103e-06, 3.3209e-06,
        2.3075e-06, 2.4137e-08, 8.4278e-06, 4.7424e-04, 1.0614e-08, 1.7783e-08,
        1.3439e-07, 3.0455e-08, 6.4612e-08, 3.4999e-08, 1.1629e-07, 2.2924e-08,
        1.4107e-07, 5.5062e-08, 5.3698e-08, 2.6498e-09, 3.9875e-09, 9.1959e-10,
        5.9642e-09, 1.0980e-08, 6.0351e-09, 2.0323e-10, 1.6535e-07, 3.8998e-12,
        1.0524e-07, 1.5661e-08, 1.1508e-08, 1.2199e-07, 5.4569e-08, 2.2328e-08,
        1.4843e-08, 2.6579e-08, 6.3592e-08, 2.0665e-07, 4.3458e-08, 1.3961